# Filtered Area Analysis EDA

This notebook analyzes:

- `classify_output/predictions.csv`
- `classify_output/filtered_areas.csv`
- `classify_output/frog_aggregated_metrics.csv`

Goals:

- Validate output quality (missing values, duplicates, join consistency).
- Summarize prediction outcomes and **classification yield** per frog/image.
- Analyze filtered-area distributions and **reference intervals**.
- Quantify within- vs between-frog variation (ICC).
- Explore morphology correlations, shape–size coupling, and nucleus scaling.
- Export publication-ready per-frog tables and regression summaries.
- Provide a final concise **Key Findings** summary.


## 1) Configuration


In [ ]:
from pathlib import Path

# Input files (override if needed)
PREDICTIONS_CSV = Path("../classify_output/predictions.csv")
FILTERED_AREAS_CSV = Path("../classify_output/filtered_areas.csv")
FROG_AGG_CSV = Path("../classify_output/frog_aggregated_metrics.csv")

# Analysis controls
TOP_N_IMAGES = 20
TOP_N_OUTLIERS = 20
TOP_N_FROGS = 20
RANDOM_SEED = 42

# Plot controls
FIG_DPI = 120


## 2) Imports and Plot Style


In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print(f"Using seaborn: {HAS_SEABORN}")


## 3) Helper Functions


In [ ]:
def ensure_file(path: Path) -> Path:
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p.resolve()}")
    return p


def resolve_join_keys(pred_df: pd.DataFrame, area_df: pd.DataFrame) -> list[str]:
    preferred = ["image_path", "mask_index"]
    keys = [k for k in preferred if k in pred_df.columns and k in area_df.columns]
    if keys:
        return keys
    if "mask_index" in pred_df.columns and "mask_index" in area_df.columns:
        return ["mask_index"]
    return []


def numeric_cols(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]


def summarize_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    if not cols:
        return pd.DataFrame()
    return df[cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


def iqr_outlier_mask(s: pd.Series) -> tuple[pd.Series, float, float]:
    s = pd.to_numeric(s, errors="coerce")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return (s < lo) | (s > hi), float(lo), float(hi)


def has_col(df: pd.DataFrame, col: str) -> bool:
    return col in df.columns


## 4) Load Data


In [ ]:
pred_path = ensure_file(PREDICTIONS_CSV)
area_path = ensure_file(FILTERED_AREAS_CSV)

pred_df = pd.read_csv(pred_path)
area_df = pd.read_csv(area_path)

frog_path = Path(FROG_AGG_CSV)
if frog_path.is_file():
    frog_df = pd.read_csv(frog_path)
    print(f"Frog aggregated metrics path: {frog_path.resolve()}")
else:
    frog_df = pd.DataFrame()
    print(f"Frog aggregated metrics path not found (optional): {frog_path.resolve()}")

print(f"Predictions path: {pred_path.resolve()}")
print(f"Filtered areas path: {area_path.resolve()}")
print(f"pred_df shape: {pred_df.shape}")
print(f"area_df shape: {area_df.shape}")
print(f"frog_df shape: {frog_df.shape}")

print("\nPredictions columns:")
print(list(pred_df.columns))
print("\nFiltered areas columns:")
print(list(area_df.columns))
print("\nFrog aggregated columns:")
print(list(frog_df.columns))

display(pred_df.head())
display(area_df.head())
if not frog_df.empty:
    display(frog_df.head())


## 5) Quality Checks (Missing, Duplicates, Basic Validity)


In [ ]:
join_keys = resolve_join_keys(pred_df, area_df)
print(f"Resolved join keys: {join_keys if join_keys else 'None'}")

print("\nMissing values (predictions):")
display(pred_df.isna().sum().to_frame("missing_count").sort_values("missing_count", ascending=False).head(20))

print("\nMissing values (filtered areas):")
display(area_df.isna().sum().to_frame("missing_count").sort_values("missing_count", ascending=False).head(20))

if join_keys:
    pred_dups = int(pred_df.duplicated(subset=join_keys).sum())
    area_dups = int(area_df.duplicated(subset=join_keys).sum())
    print(f"\nDuplicate rows by join keys in predictions: {pred_dups}")
    print(f"Duplicate rows by join keys in filtered areas: {area_dups}")
else:
    print("\nNo common join keys found; duplicate-by-key check skipped.")

area_metric_candidates = ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
for col in area_metric_candidates:
    if has_col(area_df, col):
        invalid = int((pd.to_numeric(area_df[col], errors="coerce") <= 0).sum())
        print(f"Invalid/non-positive values in {col}: {invalid}")

if has_col(pred_df, "predicted_verdict"):
    print("\nPredicted verdict unique values:")
    print(sorted(pred_df["predicted_verdict"].dropna().astype(str).unique().tolist()))

if has_col(pred_df, "accepted"):
    print("\nAccepted value counts:")
    display(pred_df["accepted"].value_counts(dropna=False).rename_axis("accepted").to_frame("count"))


## 6) Predictions EDA (Global and Per-Image)


In [ ]:
if has_col(pred_df, "predicted_verdict"):
    verdict_counts = pred_df["predicted_verdict"].value_counts(dropna=False)
    verdict_pct = (verdict_counts / max(len(pred_df), 1) * 100).round(2)
    verdict_summary = pd.DataFrame({"count": verdict_counts, "percent": verdict_pct})
    print("Predicted verdict distribution:")
    display(verdict_summary)

    plt.figure(figsize=(7, 4), dpi=FIG_DPI)
    colors = ["#009E73", "#D55E00", "#CC79A7", "#4C78A8"]
    verdict_counts.plot(kind="bar", color=colors[: len(verdict_counts)])
    plt.title("Predicted Verdict Distribution")
    plt.xlabel("predicted_verdict")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

if has_col(pred_df, "image_path") and has_col(pred_df, "predicted_verdict"):
    per_img = pred_df.groupby("image_path").size().rename("n_cells").sort_values(ascending=False)
    print(f"Top {TOP_N_IMAGES} images by number of predicted cells:")
    display(per_img.head(TOP_N_IMAGES).to_frame())

    ctab = pd.crosstab(pred_df["image_path"], pred_df["predicted_verdict"])
    ctab["total"] = ctab.sum(axis=1)
    ctab = ctab.sort_values("total", ascending=False).head(TOP_N_IMAGES)
    display(ctab)

if has_col(pred_df, "accepted") and has_col(pred_df, "predicted_verdict"):
    print("Accepted x predicted_verdict:")
    display(pd.crosstab(pred_df["predicted_verdict"], pred_df["accepted"], dropna=False))


## 7) Filtered Areas EDA (Global Statistics and Distributions)


In [ ]:
area_numeric = [c for c in numeric_cols(area_df) if c != "mask_index"]
summary = summarize_numeric(area_df, area_numeric)

if summary.empty:
    print("No numeric area columns found in filtered_areas.csv")
else:
    print("Numeric summary for filtered_areas.csv")
    display(summary)

main_metrics = [
    c
    for c in ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
    if c in area_df.columns
]

if main_metrics:
    n = len(main_metrics)
    fig, axes = plt.subplots(n, 2, figsize=(12, max(3 * n, 4)), dpi=FIG_DPI)
    if n == 1:
        axes = np.array([axes])

    for i, col in enumerate(main_metrics):
        s = pd.to_numeric(area_df[col], errors="coerce").dropna()
        ax_h = axes[i, 0]
        ax_b = axes[i, 1]

        ax_h.hist(s, bins=40, color="#1f77b4", alpha=0.8)
        ax_h.set_title(f"Histogram: {col}")
        ax_h.set_xlabel(col)
        ax_h.set_ylabel("count")

        ax_b.boxplot(s, vert=False)
        ax_b.set_title(f"Boxplot: {col}")
        ax_b.set_xlabel(col)

    plt.tight_layout()
    plt.show()
else:
    print("No expected area metrics found for distribution plots.")

if has_col(area_df, "image_path"):
    agg_dict = {}
    if has_col(area_df, "mask_index"):
        agg_dict["n_good_cells"] = ("mask_index", "count")
    else:
        first_col = area_df.columns[0]
        agg_dict["n_good_cells"] = (first_col, "count")

    if has_col(area_df, "area_px"):
        agg_dict["area_px_mean"] = ("area_px", "mean")
        agg_dict["area_px_median"] = ("area_px", "median")

    per_img_area = area_df.groupby("image_path").agg(**agg_dict).sort_values("n_good_cells", ascending=False)

    print(f"Top {TOP_N_IMAGES} images by good-cell count in filtered_areas.csv")
    display(per_img_area.head(TOP_N_IMAGES))


## 8) Join Analysis: `predictions.csv` vs `filtered_areas.csv`


In [ ]:
if not join_keys:
    merged = None
    print("Join skipped: no common keys between predictions and filtered areas.")
else:
    merged = pred_df.merge(area_df, on=join_keys, how="left", indicator=True, suffixes=("", "_area"))

    merge_counts = merged["_merge"].value_counts(dropna=False).rename_axis("merge_flag").to_frame("count")
    merge_counts["percent"] = (merge_counts["count"] / len(merged) * 100).round(2)

    print("Merge coverage (predictions LEFT JOIN filtered_areas):")
    display(merge_counts)

    if has_col(merged, "predicted_verdict"):
        area_present = merged["_merge"].eq("both")
        cov_by_verdict = (
            merged.assign(area_present=area_present)
            .groupby("predicted_verdict")["area_present"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "area_match_rate"})
        )
        cov_by_verdict["area_match_rate"] = (cov_by_verdict["area_match_rate"] * 100).round(2)
        print("Area match rate by predicted_verdict (%):")
        display(cov_by_verdict)

    if has_col(merged, "accepted"):
        area_present = merged["_merge"].eq("both")
        cov_by_acc = (
            merged.assign(area_present=area_present)
            .groupby("accepted")["area_present"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "area_match_rate"})
        )
        cov_by_acc["area_match_rate"] = (cov_by_acc["area_match_rate"] * 100).round(2)
        print("Area match rate by accepted (%):")
        display(cov_by_acc)

    if has_col(merged, "predicted_verdict") and has_col(merged, "area_px"):
        plt.figure(figsize=(8, 4), dpi=FIG_DPI)
        plot_df = merged.dropna(subset=["area_px", "predicted_verdict"]).copy()
        if not plot_df.empty:
            if HAS_SEABORN:
                order = sorted(plot_df["predicted_verdict"].astype(str).unique())
                sns.boxplot(data=plot_df, x="predicted_verdict", y="area_px", order=order)
            else:
                groups = [g["area_px"].dropna().values for _, g in plot_df.groupby("predicted_verdict")]
                labels = [str(k) for k, _ in plot_df.groupby("predicted_verdict")]
                plt.boxplot(groups, labels=labels)
            plt.title("Area Distribution by Predicted Verdict (joined rows)")
            plt.xlabel("predicted_verdict")
            plt.ylabel("area_px")
            plt.tight_layout()
            plt.show()
        else:
            print("No joined area rows available for verdict-wise area boxplot.")


## 9) Outlier Analysis (IQR)


In [ ]:
outlier_reports = {}

candidate_metrics = [
    c
    for c in ["area_px", "area_um2", "major_axis_px", "minor_axis_px", "major_axis_um", "minor_axis_um"]
    if c in area_df.columns
]

if not candidate_metrics:
    print("No candidate metrics found for outlier analysis.")
else:
    for metric in candidate_metrics:
        mask, lo, hi = iqr_outlier_mask(area_df[metric])
        out_df = area_df.loc[mask].copy().sort_values(metric, ascending=False)
        outlier_reports[metric] = {
            "count": int(mask.sum()),
            "low_bound": lo,
            "high_bound": hi,
            "top": out_df.head(TOP_N_OUTLIERS),
        }

    summary_rows = []
    for metric, rep in outlier_reports.items():
        summary_rows.append(
            {
                "metric": metric,
                "outlier_count": rep["count"],
                "low_bound": rep["low_bound"],
                "high_bound": rep["high_bound"],
            }
        )

    outlier_summary = pd.DataFrame(summary_rows).sort_values("outlier_count", ascending=False)
    print("Outlier summary (IQR method):")
    display(outlier_summary)

    for metric, rep in outlier_reports.items():
        print(
            f"\nTop outliers for {metric} (n={rep['count']}, bounds=[{rep['low_bound']:.3f}, {rep['high_bound']:.3f}]):"
        )
        cols = [
            c
            for c in ["image_path", "mask_index", metric, "major_axis_px", "minor_axis_px", "area_um2"]
            if c in rep["top"].columns
        ]
        display(rep["top"][cols])


## 10) Per-Image Deep Dive


In [ ]:
if has_col(pred_df, "image_path") and has_col(pred_df, "predicted_verdict"):
    per_img_pred = pd.crosstab(pred_df["image_path"], pred_df["predicted_verdict"])
    per_img_pred["total"] = per_img_pred.sum(axis=1)
    per_img_pred = per_img_pred.sort_values("total", ascending=False)

    top_img_pred = per_img_pred.head(TOP_N_IMAGES)
    display(top_img_pred)

    stack_cols = [c for c in ["good", "bad", "rejected"] if c in top_img_pred.columns]
    if stack_cols:
        plt.figure(figsize=(12, 5), dpi=FIG_DPI)
        x = np.arange(len(top_img_pred))
        bottom = np.zeros(len(top_img_pred))
        color_map = {"good": "#009E73", "bad": "#D55E00", "rejected": "#CC79A7"}

        for col in stack_cols:
            vals = top_img_pred[col].to_numpy()
            plt.bar(x, vals, bottom=bottom, color=color_map.get(col, None), label=col)
            bottom += vals

        plt.xticks(x, top_img_pred.index, rotation=90)
        plt.ylabel("cell count")
        plt.title(f"Top {TOP_N_IMAGES} Images: Verdict Composition")
        plt.legend()
        plt.tight_layout()
        plt.show()

if has_col(area_df, "image_path") and has_col(area_df, "area_px"):
    per_img_area = area_df.groupby("image_path").agg(
        n_good_cells=("area_px", "count"),
        area_px_mean=("area_px", "mean"),
        area_px_median=("area_px", "median"),
    ).sort_values("n_good_cells", ascending=False)

    top_img_area = per_img_area.head(TOP_N_IMAGES)
    display(top_img_area)

    plt.figure(figsize=(12, 5), dpi=FIG_DPI)
    plt.bar(top_img_area.index, top_img_area["n_good_cells"], color="#4C78A8")
    plt.xticks(rotation=90)
    plt.ylabel("good-cell count")
    plt.title(f"Top {TOP_N_IMAGES} Images by Good Cells (filtered_areas)")
    plt.tight_layout()
    plt.show()


## 11) Frog-Level Aggregation EDA


In [ ]:
if frog_df.empty:
    print("frog_aggregated_metrics.csv not found or empty; frog-level analysis skipped.")
else:
    print("Frog-level summary")
    print(f"n_frogs={len(frog_df)}, total_n_images={int(pd.to_numeric(frog_df.get('n_images', pd.Series(dtype=float)), errors='coerce').sum())}, total_n_cells={int(pd.to_numeric(frog_df.get('n_cells', pd.Series(dtype=float)), errors='coerce').sum())}")
    display(frog_df.head(TOP_N_FROGS))

    support_cols = [c for c in ["n_images", "n_cells"] if c in frog_df.columns]
    if support_cols:
        support_summary = summarize_numeric(frog_df, support_cols)
        print("\nSupport statistics")
        display(support_summary)

        fig, axes = plt.subplots(1, len(support_cols), figsize=(6 * len(support_cols), 4), dpi=FIG_DPI)
        if len(support_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, support_cols):
            vals = pd.to_numeric(frog_df[col], errors="coerce").dropna()
            ax.hist(vals, bins=30, color="#4C78A8", alpha=0.85)
            ax.set_title(f"Frog-level {col} distribution")
            ax.set_xlabel(col)
            ax.set_ylabel("count")
        plt.tight_layout()
        plt.show()

    mean_cols = [c for c in frog_df.columns if c.endswith("_mean")]
    std_cols = [c for c in frog_df.columns if c.endswith("_std")]
    print(f"Mean-metric columns: {len(mean_cols)} | Std-metric columns: {len(std_cols)}")

    if mean_cols:
        mean_summary = summarize_numeric(frog_df, mean_cols)
        print("\nSummary of frog-level mean metrics")
        display(mean_summary.head(20))

        plot_means = [
            c for c in ["area_px_mean", "cell_axis_ratio_mean", "nucleus_axis_ratio_mean", "nc_ratio_mean"]
            if c in frog_df.columns
        ]
        if plot_means:
            fig, axes = plt.subplots(len(plot_means), 1, figsize=(8, 3.2 * len(plot_means)), dpi=FIG_DPI)
            if len(plot_means) == 1:
                axes = [axes]
            for ax, col in zip(axes, plot_means):
                vals = pd.to_numeric(frog_df[col], errors="coerce").dropna()
                ax.hist(vals, bins=30, color="#009E73", alpha=0.85)
                ax.set_title(f"Distribution: {col}")
                ax.set_xlabel(col)
                ax.set_ylabel("count")
            plt.tight_layout()
            plt.show()

    # Aggregation consistency checks against filtered_areas.csv
    if "frog_id" in area_df.columns and "frog_id" in frog_df.columns:
        area_work = area_df.copy()
        area_work["frog_id"] = pd.to_numeric(area_work["frog_id"], errors="coerce")
        area_work = area_work[area_work["frog_id"].notna()].copy()
        area_work["frog_id"] = area_work["frog_id"].astype(int)

        expected = area_work.groupby("frog_id", as_index=False).agg(
            n_images_expected=("image_path", "nunique"),
            n_cells_expected=("mask_index", "count"),
        )
        cmp = frog_df.merge(expected, on="frog_id", how="outer", indicator=True)
        cmp["n_images_match"] = pd.to_numeric(cmp.get("n_images", np.nan), errors="coerce") == pd.to_numeric(cmp.get("n_images_expected", np.nan), errors="coerce")
        cmp["n_cells_match"] = pd.to_numeric(cmp.get("n_cells", np.nan), errors="coerce") == pd.to_numeric(cmp.get("n_cells_expected", np.nan), errors="coerce")

        print("\nAggregation completeness check")
        display(cmp[["frog_id", "_merge", "n_images", "n_images_expected", "n_images_match", "n_cells", "n_cells_expected", "n_cells_match"]].head(TOP_N_FROGS))

        mismatches = cmp[(cmp["_merge"] != "both") | (~cmp["n_images_match"]) | (~cmp["n_cells_match"])].copy()
        print(f"Mismatched frog rows: {len(mismatches)}")
        if len(mismatches) > 0:
            display(mismatches.head(TOP_N_FROGS))
    else:
        print("frog_id column missing in area_df or frog_df; consistency check skipped.")


## 12) Report-Style Biology Figures

Publication-ready biology distributions and per-frog plots that mirror
`notebooks/make_report_figures.py`. These fill gaps the earlier EDA
sections don't cover:

- Per-cell **nucleus** size and **N/C ratio** distributions.
- Per-frog **boxplots** for cell and nucleus area (top-40 frogs,
  sorted by median cell area).
- Per-frog **scatter** of mean cell area vs. number of good cells.
- **Outlier frogs** (5 smallest and 5 largest mean cell area)
  highlighted against the full dataset.

The cell-area and cell-diameter panels are re-plotted in report style
(with median + IQR overlays) for consistency with the rest of the report.


In [ ]:
BIO_COLOR_CELL = "#2a9d8f"
BIO_COLOR_NUCLEUS = "#264653"
BIO_COLOR_HIGHLIGHT = "#e76f51"
BIO_COLOR_HIGHLIGHT_LARGE = "#1d4ed8"


def hist_with_summary(
    ax: plt.Axes,
    values,
    color: str,
    title: str,
    xlabel: str,
    bins: int = 40,
    xlim_percentiles: tuple[float, float] | None = None,
) -> None:
    """Histogram with median line and IQR band overlay.

    If xlim_percentiles is set, e.g. (1, 99), the x-axis is zoomed to that
    percentile range so the dominant bulk of the distribution is visible.
    """
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy()
    plot_values = values
    hist_range = None

    if len(values) and xlim_percentiles is not None:
        lo_p, hi_p = xlim_percentiles
        lo, hi = np.quantile(values, [lo_p / 100.0, hi_p / 100.0])
        span = hi - lo
        pad = span * 0.05 if span > 0 else 0.5
        lo, hi = lo - pad, hi + pad
        hist_range = (lo, hi)
        plot_values = values[(values >= lo) & (values <= hi)]

    if len(plot_values):
        ax.hist(
            plot_values,
            bins=bins,
            range=hist_range,
            color=color,
            alpha=0.85,
            edgecolor="white",
            linewidth=0.5,
        )
        if hist_range is not None:
            ax.set_xlim(hist_range)

    if len(values):
        med = float(np.median(values))
        q1, q3 = np.quantile(values, [0.25, 0.75])
        from matplotlib.lines import Line2D

        ax.axvline(med, color="#222", linestyle="--", linewidth=1.3, label=f"median={med:.2f}")
        ax.axvspan(q1, q3, color="#222", alpha=0.08, label=f"IQR {q1:.2f}–{q3:.2f}")
        handles, labels = ax.get_legend_handles_labels()
        if xlim_percentiles is not None:
            n_out = len(values) - len(plot_values)
            if n_out:
                handles.append(
                    Line2D([0], [0], linestyle="none", marker="none", color="none")
                )
                labels.append(
                    f"{n_out} outside {xlim_percentiles[0]:.0f}–{xlim_percentiles[1]:.0f}%"
                )
        ax.legend(handles, labels, loc="upper right", fontsize=9, framealpha=0.92)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of cells")


### 12.1) Cell area distribution (report style)


In [ ]:
if "area_um2" in area_df.columns:
    fig, ax = plt.subplots(figsize=(7, 4.6), dpi=FIG_DPI)
    hist_with_summary(
        ax, area_df["area_um2"], BIO_COLOR_CELL,
        "Cell area (µm²)", "area_um2",
    )
    fig.suptitle("Cell area distribution (good cells only)", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
else:
    print("area_um2 column not available; skipping cell area distribution.")


### 12.2) Cell diameter distribution (report style)


In [ ]:
if {"major_axis_um", "minor_axis_um"}.issubset(area_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), dpi=FIG_DPI)
    hist_with_summary(
        axes[0], area_df["major_axis_um"], BIO_COLOR_CELL,
        "Cell long diameter (µm)", "major_axis_um",
    )
    hist_with_summary(
        axes[1], area_df["minor_axis_um"], BIO_COLOR_CELL,
        "Cell short diameter (µm)", "minor_axis_um",
    )
    fig.suptitle("Cell diameter distribution (good cells only)", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
else:
    print("major_axis_um / minor_axis_um columns not available; skipping diameter distribution.")


### 12.3) Nucleus size distribution


In [ ]:
nuc_cols = ["nucleus_area_um2", "nucleus_major_axis_um", "nucleus_minor_axis_um"]
nuc_missing = [c for c in nuc_cols if c not in area_df.columns]

if not nuc_missing:
    n_matched = int(pd.to_numeric(area_df["nucleus_area_um2"], errors="coerce").notna().sum())
    print(f"Cells with a matched nucleus: {n_matched} / {len(area_df)}")

    # Zoom to central bulk: full-range axes hide the dominant peak (long outlier tails).
    nuc_zoom_percentiles = (1, 99)
    nuc_zoom_bins = 50

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), dpi=FIG_DPI)
    hist_with_summary(
        axes[0], area_df["nucleus_area_um2"], BIO_COLOR_NUCLEUS,
        "Nucleus area (µm²)", "nucleus_area_um2",
        bins=nuc_zoom_bins, xlim_percentiles=nuc_zoom_percentiles,
    )
    hist_with_summary(
        axes[1], area_df["nucleus_major_axis_um"], BIO_COLOR_NUCLEUS,
        "Nucleus long diameter (µm)", "nucleus_major_axis_um",
        bins=nuc_zoom_bins, xlim_percentiles=nuc_zoom_percentiles,
    )
    hist_with_summary(
        axes[2], area_df["nucleus_minor_axis_um"], BIO_COLOR_NUCLEUS,
        "Nucleus short diameter (µm)", "nucleus_minor_axis_um",
        bins=nuc_zoom_bins, xlim_percentiles=nuc_zoom_percentiles,
    )
    fig.suptitle(
        "Nucleus size distribution (matched cells; x-axis zoomed to 1st–99th percentile)",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
else:
    print(f"Missing nucleus columns: {nuc_missing}; skipping nucleus distribution.")


### 12.4) N/C ratio distribution


In [ ]:
if "nc_ratio" in area_df.columns:
    fig, ax = plt.subplots(figsize=(9, 4.8), dpi=FIG_DPI)
    hist_with_summary(
        ax, area_df["nc_ratio"], BIO_COLOR_NUCLEUS,
        "Nucleus-to-cell area ratio (N/C)", "nc_ratio",
        bins=50,
        xlim_percentiles=(1, 99),
    )
    fig.suptitle(
        "N/C ratio — nucleus share of cell area (x-axis zoomed to 1st–99th percentile)",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print("nc_ratio column not available; skipping N/C ratio distribution.")


### 12.5) Per-frog cell-area and nucleus-area boxplots

Top-40 frogs by number of good cells, sorted by median cell area. The
nucleus-area boxplot below uses the **same frog ordering** as the
cell-area boxplot so the two figures can be read side by side.


In [ ]:
TOP_K_FROGS_FOR_BOXPLOT = 40


def _per_frog_boxplot(
    area_df: pd.DataFrame,
    value_col: str,
    frogs_sorted: list,
    color: str,
    ylabel: str,
    title: str,
) -> None:
    sub = area_df[area_df["frog_id"].isin(frogs_sorted)]
    data = [
        sub.loc[sub["frog_id"] == fid, value_col].dropna().to_numpy()
        for fid in frogs_sorted
    ]
    sample_counts = [len(d) for d in data]

    fig, ax = plt.subplots(figsize=(14, 5.5), dpi=FIG_DPI)
    bp = ax.boxplot(
        data,
        positions=np.arange(len(frogs_sorted)),
        widths=0.6,
        patch_artist=True,
        showfliers=False,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        patch.set_edgecolor("#0f172a")
    for med in bp["medians"]:
        med.set_color("#0f172a")
        med.set_linewidth(1.5)

    ax.set_xticks(np.arange(len(frogs_sorted)))
    ax.set_xticklabels(
        [f"{fid}\n(n={n})" for fid, n in zip(frogs_sorted, sample_counts)],
        rotation=60,
        ha="right",
        fontsize=7,
    )
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()


if {"frog_id", "area_um2"}.issubset(area_df.columns):
    counts = area_df.groupby("frog_id").size().sort_values(ascending=False)
    top_frogs = counts.head(TOP_K_FROGS_FOR_BOXPLOT).index.tolist()

    sub_top = area_df[area_df["frog_id"].isin(top_frogs)]
    medians = sub_top.groupby("frog_id")["area_um2"].median().sort_values()
    frogs_sorted = medians.index.tolist()

    _per_frog_boxplot(
        area_df=area_df,
        value_col="area_um2",
        frogs_sorted=frogs_sorted,
        color=BIO_COLOR_CELL,
        ylabel="Cell area (µm²)",
        title=(
            f"Per-frog cell-area distribution (top-{len(frogs_sorted)} frogs "
            f"by cell count, sorted by median cell area)"
        ),
    )

    if "nucleus_area_um2" in area_df.columns:
        _per_frog_boxplot(
            area_df=area_df,
            value_col="nucleus_area_um2",
            frogs_sorted=frogs_sorted,
            color=BIO_COLOR_NUCLEUS,
            ylabel="Nucleus area (µm²)",
            title=(
                f"Per-frog nucleus-area distribution (same {len(frogs_sorted)} frogs, "
                f"same ordering as the cell-area boxplot)"
            ),
        )
    else:
        print("nucleus_area_um2 not available; skipping nucleus boxplot.")
else:
    print("frog_id / area_um2 not available; skipping per-frog boxplots.")


### 12.6) Per-frog scatter: mean cell area vs. number of good cells

Each point is one frog. Error bars are the within-frog SD of cell area.
The dashed line and shaded band show the across-frog median and
mean ± 1 SD of per-frog mean cell area.


In [ ]:
if {"area_um2_mean", "n_cells"}.issubset(frog_df.columns):
    df_scatter = frog_df.dropna(subset=["area_um2_mean"]).copy()

    fig, ax = plt.subplots(figsize=(11, 5.5), dpi=FIG_DPI)
    ax.errorbar(
        df_scatter["n_cells"],
        df_scatter["area_um2_mean"],
        yerr=df_scatter.get("area_um2_std"),
        fmt="o",
        color=BIO_COLOR_CELL,
        ecolor="#8ab8b3",
        elinewidth=0.8,
        capsize=2,
        markersize=5,
        alpha=0.85,
    )

    overall_median = df_scatter["area_um2_mean"].median()
    overall_mean = df_scatter["area_um2_mean"].mean()
    overall_std = df_scatter["area_um2_mean"].std()
    ax.axhline(
        overall_median, color="#1a4f49", linestyle="--", linewidth=1.2,
        label=f"across-frog median = {overall_median:.1f} µm²",
    )
    ax.axhspan(
        overall_mean - overall_std, overall_mean + overall_std,
        color="#1a4f49", alpha=0.08, label="across-frog mean ± 1 SD",
    )

    ax.set_xlabel("Number of good cells measured per frog")
    ax.set_ylabel("Mean cell area per frog (µm²)")
    ax.set_title("Cross-frog variability of mean cell area (error bars = within-frog SD)")
    ax.legend(loc="best", fontsize=10)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print("area_um2_mean / n_cells not available in frog_df; skipping per-frog scatter.")


### 12.7) Outlier frogs (5 smallest and 5 largest mean cell area)

- **Left**: per-frog ranking by mean cell area. Orange bars mark the
  5 frogs with the smallest mean cell area; blue bars mark the 5
  largest. Dashed line is the across-frog median.
- **Right**: cell-area histogram of the full good-cell dataset (gray),
  with the same outlier frogs overlaid so the extreme frogs can be
  compared against the population.


In [ ]:
if "area_um2_mean" in frog_df.columns and "area_um2" in area_df.columns:
    df_ranked = (
        frog_df.dropna(subset=["area_um2_mean"])
        .copy()
        .sort_values("area_um2_mean")
    )
    small = df_ranked.head(5)["frog_id"].tolist()
    large = df_ranked.tail(5)["frog_id"].tolist()
    overall_median = df_ranked["area_um2_mean"].median()

    df_ranked = df_ranked.reset_index(drop=True)
    df_ranked["rank"] = df_ranked.index + 1
    n_frogs = len(df_ranked)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2), dpi=FIG_DPI)

    # Rank scatter: readable for all frogs (barh with ~479 y-labels is illegible).
    ax1.scatter(
        df_ranked["rank"],
        df_ranked["area_um2_mean"],
        s=16,
        c="#cbd5e1",
        alpha=0.7,
        edgecolors="none",
        label=f"all frogs (n={n_frogs})",
    )
    small_mask = df_ranked["frog_id"].isin(small)
    large_mask = df_ranked["frog_id"].isin(large)
    ax1.scatter(
        df_ranked.loc[small_mask, "rank"],
        df_ranked.loc[small_mask, "area_um2_mean"],
        s=48,
        c=BIO_COLOR_HIGHLIGHT,
        edgecolors="white",
        linewidths=0.6,
        zorder=3,
        label="smallest 5",
    )
    ax1.scatter(
        df_ranked.loc[large_mask, "rank"],
        df_ranked.loc[large_mask, "area_um2_mean"],
        s=48,
        c=BIO_COLOR_HIGHLIGHT_LARGE,
        edgecolors="white",
        linewidths=0.6,
        zorder=3,
        label="largest 5",
    )
    for _, row in df_ranked[small_mask | large_mask].iterrows():
        ax1.annotate(
            str(int(row["frog_id"])),
            (row["rank"], row["area_um2_mean"]),
            textcoords="offset points",
            xytext=(4, 4),
            fontsize=8,
            color="#333",
        )
    ax1.axhline(
        overall_median,
        color="#1a4f49",
        linestyle="--",
        linewidth=1.2,
        label=f"median = {overall_median:.1f} µm²",
    )
    ax1.set_xlabel("Rank by mean cell area (1 = smallest)")
    ax1.set_ylabel("Mean cell area per frog (µm²)")
    ax1.set_title("Per-frog ranking (orange = smallest 5, blue = largest 5)")
    ax1.legend(loc="lower right", fontsize=9)
    ax1.grid(alpha=0.2)

    bins = np.linspace(
        float(area_df["area_um2"].quantile(0.01)),
        float(area_df["area_um2"].quantile(0.99)),
        60,
    )
    ax2.hist(
        area_df["area_um2"].dropna(),
        bins=bins,
        color="#cbd5e1",
        alpha=0.6,
        label="all good cells",
    )
    for fid in small + large:
        color = BIO_COLOR_HIGHLIGHT if fid in small else BIO_COLOR_HIGHLIGHT_LARGE
        sub = area_df.loc[area_df["frog_id"] == fid, "area_um2"].dropna()
        if not len(sub):
            continue
        ax2.hist(sub, bins=bins, alpha=0.55, color=color, label=f"frog {fid} (n={len(sub)})")
    ax2.set_xlabel("Cell area (µm²)")
    ax2.set_ylabel("Number of cells")
    ax2.set_title("Area distribution: outlier frogs vs. the whole dataset")
    ax2.legend(fontsize=8, loc="upper right")
    ax2.grid(axis="y", alpha=0.2)

    fig.suptitle("Outlier frogs — worth a manual check", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

    print("Smallest 5 frogs by mean cell area:")
    display(df_ranked.head(5)[["frog_id", "n_cells", "area_um2_mean", "area_um2_std"]])
    print("\nLargest 5 frogs by mean cell area:")
    display(df_ranked.tail(5)[["frog_id", "n_cells", "area_um2_mean", "area_um2_std"]])
else:
    print("Required columns missing; skipping outlier-frogs figure.")


## 13) Cell vs Nucleus — Regression Analysis

We test how nucleus size scales with cell size using three log-log regressions:

1. **Area vs area**: `log(nucleus_area_um2) ~ log(area_um2)`
2. **Major axis vs major axis** (long diameters)
3. **Minor axis vs minor axis** (short diameters)

For each, we fit:

- a **pooled OLS** (`statsmodels.OLS`) with analytical 95% CIs and an
  additional **bootstrap 95% CI** on the slope (1000 cell-level resamples),
- a **mixed-effects model** (`statsmodels.MixedLM`) with a random intercept
  per `frog_id`, so the slope is estimated after removing per-frog level
  differences.

We then save residual diagnostics (residuals-vs-fitted, QQ), list the
top-10 cells with the largest absolute studentized residuals, and export
both the summary and the outliers to `classify_output/regression/`.

**Slope interpretation (allometry).**
- slope ≈ 1 → **isometric** scaling (nucleus area grows in proportion to cell area)
- slope < 1 → nucleus grows **slower** than the cell (relatively smaller in big cells)
- slope > 1 → nucleus grows **faster** than the cell (relatively larger in big cells)

The main scatter plot is colored by `frog_id` (top-40 frogs by cell count
get distinct colors; the remaining frogs are pooled into a light-grey
"other" group) so per-frog clustering is visible.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scipy.stats as sps
import statsmodels.api as sm

REGRESSION_OUT_DIR = Path("../classify_output/regression")
REGRESSION_OUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
TOP_N_OUTLIERS = 10
TOP_N_FROGS_FOR_COLOR = 40
RNG = np.random.default_rng(42)


def _ols_fit_log(df: pd.DataFrame, x_col: str, y_col: str) -> dict:
    """Pooled OLS in log-log space + analytical and bootstrap CIs."""
    work = df[[x_col, y_col, "frog_id"]].dropna()
    work = work[(work[x_col] > 0) & (work[y_col] > 0)]
    n = len(work)
    if n < 10:
        return {"n": n, "ok": False, "reason": "not enough rows"}

    x = np.log(work[x_col].to_numpy(dtype=float))
    y = np.log(work[y_col].to_numpy(dtype=float))
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    intercept, slope = float(model.params[0]), float(model.params[1])
    ci = model.conf_int(alpha=0.05)
    intercept_ci = (float(ci[0, 0]), float(ci[0, 1]))
    slope_ci = (float(ci[1, 0]), float(ci[1, 1]))

    boot_slopes = np.empty(N_BOOTSTRAP, dtype=float)
    idx = np.arange(n)
    for b in range(N_BOOTSTRAP):
        sample = RNG.choice(idx, size=n, replace=True)
        Xb = sm.add_constant(x[sample])
        boot_slopes[b] = sm.OLS(y[sample], Xb).fit().params[1]
    slope_ci_boot = (float(np.quantile(boot_slopes, 0.025)),
                     float(np.quantile(boot_slopes, 0.975)))

    influence = model.get_influence()
    resid_studentized = influence.resid_studentized_internal
    work = work.assign(
        log_x=x, log_y=y,
        fitted=model.fittedvalues,
        resid=model.resid,
        resid_studentized=resid_studentized,
    )
    return {
        "ok": True,
        "n": n,
        "n_frogs": int(work["frog_id"].nunique()),
        "model": model,
        "x_col": x_col, "y_col": y_col,
        "slope": slope, "intercept": intercept,
        "slope_ci": slope_ci, "intercept_ci": intercept_ci,
        "slope_ci_bootstrap": slope_ci_boot,
        "r2": float(model.rsquared),
        "rmse": float(np.sqrt(np.mean(model.resid ** 2))),
        "data": work,
    }


def _mixedlm_fit_log(df: pd.DataFrame, x_col: str, y_col: str) -> dict:
    """Random-intercept-per-frog mixed-effects fit in log-log space."""
    work = df[[x_col, y_col, "frog_id"]].dropna()
    work = work[(work[x_col] > 0) & (work[y_col] > 0)]
    if len(work) < 10 or work["frog_id"].nunique() < 2:
        return {"ok": False, "reason": "not enough data / frogs"}
    work = work.assign(
        log_x=np.log(work[x_col].to_numpy(dtype=float)),
        log_y=np.log(work[y_col].to_numpy(dtype=float)),
    )
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            md = sm.MixedLM.from_formula(
                "log_y ~ log_x", groups="frog_id", data=work
            )
            res = md.fit(method="lbfgs", reml=True)
    except Exception as exc:
        return {"ok": False, "reason": f"MixedLM failed: {exc}"}
    intercept = float(res.fe_params["Intercept"])
    slope = float(res.fe_params["log_x"])
    ci = res.conf_int(alpha=0.05)
    slope_ci = (float(ci.loc["log_x", 0]), float(ci.loc["log_x", 1]))
    intercept_ci = (float(ci.loc["Intercept", 0]), float(ci.loc["Intercept", 1]))
    return {
        "ok": True,
        "n": len(work),
        "n_frogs": int(work["frog_id"].nunique()),
        "slope": slope, "intercept": intercept,
        "slope_ci": slope_ci, "intercept_ci": intercept_ci,
    }


def _interpret_slope(slope: float, lo: float, hi: float) -> str:
    if lo > 1.0:
        return "positively allometric (nucleus grows faster than cell)"
    if hi < 1.0:
        return "negatively allometric (nucleus grows slower than cell)"
    return "consistent with isometric scaling (slope CI contains 1.0)"


def _plot_regression(fit: dict, mixed: dict | None, label: str, ax) -> None:
    df = fit["data"]
    counts = df.groupby("frog_id").size().sort_values(ascending=False)
    top_frogs = counts.head(TOP_N_FROGS_FOR_COLOR).index.tolist()
    cmap = plt.colormaps.get_cmap("tab20")
    color_map = {fid: cmap(i % cmap.N) for i, fid in enumerate(top_frogs)}

    is_top = df["frog_id"].isin(top_frogs)
    other = df.loc[~is_top]
    ax.scatter(
        other["log_x"], other["log_y"],
        s=10, c="#cbd5e1", alpha=0.55, edgecolors="none",
        label=f"other frogs (n={len(other)})",
    )
    for fid in top_frogs:
        sub = df.loc[df["frog_id"] == fid]
        ax.scatter(
            sub["log_x"], sub["log_y"],
            s=14, color=color_map[fid], alpha=0.85, edgecolors="none",
        )

    xx = np.linspace(df["log_x"].min(), df["log_x"].max(), 100)
    yy_pooled = fit["intercept"] + fit["slope"] * xx
    ax.plot(xx, yy_pooled, color="#0f172a", lw=2.2,
            label=f"pooled OLS  slope={fit['slope']:.3f}  R²={fit['r2']:.3f}")
    if mixed and mixed.get("ok"):
        yy_mx = mixed["intercept"] + mixed["slope"] * xx
        ax.plot(xx, yy_mx, color="#e76f51", lw=2.0, ls="--",
                label=f"mixed-effects  slope={mixed['slope']:.3f}")
    ax.plot(xx, xx + (fit["intercept"] + fit["slope"] * xx.mean() - xx.mean()),
            color="#94a3b8", lw=1.0, ls=":", alpha=0.0)

    ax.set_xlabel(f"log({fit['x_col']})")
    ax.set_ylabel(f"log({fit['y_col']})")
    ax.set_title(f"{label}\n(n={fit['n']} cells, {fit['n_frogs']} frogs)")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", fontsize=9, frameon=True)


def _plot_diagnostics(fit: dict, label: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    df = fit["data"]
    axes[0].scatter(df["fitted"], df["resid_studentized"], s=8, alpha=0.55, color="#2a9d8f")
    axes[0].axhline(0, color="#0f172a", lw=1)
    for thr, color in [(2.0, "#e76f51"), (3.0, "#b91c1c")]:
        axes[0].axhline(thr, color=color, lw=0.8, ls="--", alpha=0.7)
        axes[0].axhline(-thr, color=color, lw=0.8, ls="--", alpha=0.7)
    axes[0].set_xlabel("fitted log(y)")
    axes[0].set_ylabel("studentized residual")
    axes[0].set_title(f"Residuals vs fitted — {label}")
    axes[0].grid(alpha=0.25)

    sm.qqplot(df["resid"], line="45", ax=axes[1], markerfacecolor="#2a9d8f",
              markeredgecolor="none", alpha=0.6)
    axes[1].set_title(f"QQ plot of residuals — {label}")
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    plt.show()


def _top_outliers(fit: dict, top_n: int) -> pd.DataFrame:
    df = fit["data"].copy()
    keep_cols = [c for c in ["frog_id", "image_path", "mask_index",
                              fit["x_col"], fit["y_col"],
                              "fitted", "resid", "resid_studentized"]
                 if c in df.columns or c in ("fitted", "resid", "resid_studentized")]
    if "image_path" not in df.columns or "mask_index" not in df.columns:
        for missing in ("image_path", "mask_index"):
            if missing not in df.columns:
                df[missing] = pd.NA
                if missing not in keep_cols:
                    keep_cols.append(missing)
    df = df.assign(abs_resid=np.abs(df["resid_studentized"]))
    df = df.sort_values("abs_resid", ascending=False).head(top_n)
    df["regression"] = f"{fit['y_col']}_vs_{fit['x_col']}"
    return df[["regression", *keep_cols, "abs_resid"]].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Run all three regressions on filtered_areas.csv
# ---------------------------------------------------------------------------

regression_targets = [
    ("area_um2", "nucleus_area_um2",
     "Nucleus area vs cell area (log-log)"),
    ("major_axis_um", "nucleus_major_axis_um",
     "Nucleus long diameter vs cell long diameter (log-log)"),
    ("minor_axis_um", "nucleus_minor_axis_um",
     "Nucleus short diameter vs cell short diameter (log-log)"),
]

summary_rows = []
outlier_frames = []
fits = {}

for x_col, y_col, label in regression_targets:
    print("=" * 78)
    print(f"REGRESSION: log({y_col})  ~  log({x_col})")
    print(label)
    print("=" * 78)

    fit = _ols_fit_log(area_df, x_col, y_col)
    if not fit.get("ok"):
        print(f"  skipped: {fit.get('reason')}")
        continue
    mixed = _mixedlm_fit_log(area_df, x_col, y_col)

    print(f"  n_cells = {fit['n']:,}    n_frogs = {fit['n_frogs']}")
    print(f"  pooled OLS  : slope = {fit['slope']:.4f}   "
          f"95% CI = [{fit['slope_ci'][0]:.4f}, {fit['slope_ci'][1]:.4f}]")
    print(f"               intercept = {fit['intercept']:.4f}   "
          f"95% CI = [{fit['intercept_ci'][0]:.4f}, {fit['intercept_ci'][1]:.4f}]")
    print(f"               R² = {fit['r2']:.4f}    RMSE(log) = {fit['rmse']:.4f}")
    print(f"               bootstrap slope 95% CI = "
          f"[{fit['slope_ci_bootstrap'][0]:.4f}, {fit['slope_ci_bootstrap'][1]:.4f}]")
    if mixed.get("ok"):
        print(f"  mixed-effects (random intercept per frog):")
        print(f"               slope = {mixed['slope']:.4f}   "
              f"95% CI = [{mixed['slope_ci'][0]:.4f}, {mixed['slope_ci'][1]:.4f}]")
        interp_slope, lo, hi = mixed["slope"], *mixed["slope_ci"]
    else:
        print(f"  mixed-effects fit unavailable: {mixed.get('reason')}")
        interp_slope, lo, hi = fit["slope"], *fit["slope_ci"]
    print(f"  interpretation: {_interpret_slope(interp_slope, lo, hi)}")

    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_regression(fit, mixed, label, ax)
    plt.show()

    _plot_diagnostics(fit, label)

    out = _top_outliers(fit, TOP_N_OUTLIERS)
    print(f"\nTop-{TOP_N_OUTLIERS} outliers by |studentized residual|:")
    display_cols = [c for c in ["frog_id", "image_path", "mask_index",
                                 x_col, y_col, "resid_studentized"] if c in out.columns]
    print(out[display_cols].to_string(index=False))
    outlier_frames.append(out)

    summary_rows.append({
        "regression": f"log({y_col}) ~ log({x_col})",
        "n_cells": fit["n"],
        "n_frogs": fit["n_frogs"],
        "ols_slope": fit["slope"],
        "ols_slope_ci_lo": fit["slope_ci"][0],
        "ols_slope_ci_hi": fit["slope_ci"][1],
        "ols_slope_ci_bootstrap_lo": fit["slope_ci_bootstrap"][0],
        "ols_slope_ci_bootstrap_hi": fit["slope_ci_bootstrap"][1],
        "ols_intercept": fit["intercept"],
        "ols_intercept_ci_lo": fit["intercept_ci"][0],
        "ols_intercept_ci_hi": fit["intercept_ci"][1],
        "ols_r2": fit["r2"],
        "ols_rmse_log": fit["rmse"],
        "mixedlm_slope": mixed["slope"] if mixed.get("ok") else np.nan,
        "mixedlm_slope_ci_lo": mixed["slope_ci"][0] if mixed.get("ok") else np.nan,
        "mixedlm_slope_ci_hi": mixed["slope_ci"][1] if mixed.get("ok") else np.nan,
        "mixedlm_intercept": mixed["intercept"] if mixed.get("ok") else np.nan,
    })
    fits[(x_col, y_col)] = (fit, mixed)

# ---------------------------------------------------------------------------
# Persist results to disk
# ---------------------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)
summary_path = REGRESSION_OUT_DIR / "regression_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("\nSaved regression summary to:", summary_path)
display(summary_df.round(4))

if outlier_frames:
    outliers_df = pd.concat(outlier_frames, ignore_index=True)
    outliers_path = REGRESSION_OUT_DIR / "regression_outliers.csv"
    outliers_df.to_csv(outliers_path, index=False)
    print("Saved regression outliers to:", outliers_path)

# ---------------------------------------------------------------------------
# Optional: export the area-vs-area scatter as a figure for the report
# ---------------------------------------------------------------------------

key = ("area_um2", "nucleus_area_um2")
if key in fits:
    fit, mixed = fits[key]
    fig_dir = Path("./figures")
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(11, 6))
    _plot_regression(fit, mixed, "Nucleus area vs cell area (log-log)", ax)
    fig.tight_layout()
    fig.savefig(fig_dir / "NucleusVsCellRegression.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print("Saved report figure to:", (fig_dir / "NucleusVsCellRegression.png").resolve())


## 15) Classification Yield & Sampling Bias

Morphology is measured only on **good** cells. This section summarizes classifier yield per frog and per image so size summaries can be interpreted with sampling context.


In [ ]:
if has_col(pred_df, "predicted_verdict") and has_col(pred_df, "frog_id"):
    yield_frog = (
        pred_df.groupby("frog_id", as_index=False)
        .agg(
            n_total=("mask_index", "count"),
            n_good=("predicted_verdict", lambda s: (s == "good").sum()),
            n_bad=("predicted_verdict", lambda s: (s == "bad").sum()),
            n_rejected=("predicted_verdict", lambda s: (s == "rejected").sum()),
        )
        .assign(good_fraction=lambda d: d["n_good"] / d["n_total"].clip(lower=1))
    )

    print("Per-frog classification yield (lowest / highest good_fraction):")
    display(yield_frog.sort_values("good_fraction").head(10))
    display(yield_frog.sort_values("good_fraction", ascending=False).head(10))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=FIG_DPI)

    ax = axes[0]
    yf = yield_frog.sort_values("frog_id").head(80)
    bottom = np.zeros(len(yf))
    x = np.arange(len(yf))
    for verdict, col in [("good", "n_good"), ("bad", "n_bad"), ("rejected", "n_rejected")]:
        vals = yf[col].to_numpy(dtype=float)
        colors = {"good": "#009E73", "bad": "#D55E00", "rejected": "#CC79A7"}
        ax.bar(x, vals, bottom=bottom, color=colors[verdict], label=verdict, width=1.0)
        bottom += vals
    ax.set_xlim(-0.5, len(yf) - 0.5)
    ax.set_xticks([])
    ax.set_ylabel("Cell count")
    ax.set_title(f"Stacked verdicts (first {len(yf)} frogs by frog_id)")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.2)

    ax = axes[1]
    ax.scatter(yield_frog["n_good"], yield_frog["good_fraction"], s=22, alpha=0.55, c="#4C78A8", edgecolors="none")
    if not frog_df.empty and "area_um2_mean" in frog_df.columns:
        y2 = yield_frog.merge(frog_df[["frog_id", "area_um2_mean"]], on="frog_id", how="left")
        r = y2["good_fraction"].corr(y2["area_um2_mean"])
        ax.text(0.03, 0.97, f"corr(good_fraction, mean area) = {r:.3f}", transform=ax.transAxes, va="top", fontsize=9)
    ax.set_xlabel("Number of good cells per frog")
    ax.set_ylabel("Good fraction")
    ax.set_title("Yield vs. good-cell count")
    ax.grid(alpha=0.2)

    fig.suptitle("Classifier yield — morphology is a subset of all detections", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

    if has_col(pred_df, "image_path"):
        yield_img = (
            pred_df.groupby("image_path", as_index=False)
            .agg(n_total=("mask_index", "count"), n_good=("predicted_verdict", lambda s: (s == "good").sum()))
            .assign(good_fraction=lambda d: d["n_good"] / d["n_total"].clip(lower=1))
        )
        print(
            f"Per-image good fraction: median {yield_img['good_fraction'].median():.3f}; "
            f"images with 0 good: {(yield_img['n_good'] == 0).sum()}"
        )
else:
    print("predictions.csv missing verdict or frog_id; skipping yield analysis.")


## 16) Reference Intervals & Equivalent Diameter

Normative percentile ranges for good-cell morphology and equivalent circular diameter from area.


In [ ]:
ANALYSIS_OUT_DIR = Path("../classify_output/analysis")
ANALYSIS_OUT_DIR.mkdir(parents=True, exist_ok=True)

REF_PERCENTILES = [2.5, 5, 25, 50, 75, 95, 97.5]
REF_METRICS = [
    ("area_um2", "Cell area (µm²)"),
    ("equiv_diameter_um", "Cell equivalent diameter (µm)"),
    ("major_axis_um", "Cell long axis (µm)"),
    ("minor_axis_um", "Cell short axis (µm)"),
    ("nucleus_area_um2", "Nucleus area (µm²)"),
    ("nc_ratio", "N/C area ratio"),
    ("cell_axis_ratio", "Cell axis ratio"),
]

area_ref = area_df.copy()
if "area_um2" in area_ref.columns:
    a = pd.to_numeric(area_ref["area_um2"], errors="coerce")
    area_ref["equiv_diameter_um"] = 2 * np.sqrt(a / np.pi)
if "nucleus_area_um2" in area_ref.columns:
    na = pd.to_numeric(area_ref["nucleus_area_um2"], errors="coerce")
    area_ref["nucleus_equiv_diameter_um"] = 2 * np.sqrt(na / np.pi)

rows = []
for col, label in REF_METRICS:
    if col not in area_ref.columns:
        continue
    s = pd.to_numeric(area_ref[col], errors="coerce").dropna()
    if s.empty:
        continue
    pct = s.quantile([p / 100 for p in REF_PERCENTILES])
    row = {"metric": col, "label": label, "n": int(len(s))}
    for p, v in zip(REF_PERCENTILES, pct):
        row[f"p{p}"] = float(v)
    rows.append(row)

ref_df = pd.DataFrame(rows)
if not ref_df.empty:
    print("Reference intervals for good cells (percentiles):")
    display(ref_df.round(4))
    ref_path = ANALYSIS_OUT_DIR / "reference_intervals.csv"
    ref_df.to_csv(ref_path, index=False)
    print("Saved:", ref_path.resolve())

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(ref_df))), dpi=FIG_DPI)
    y = np.arange(len(ref_df))
    med = ref_df["p50"].to_numpy()
    err_lo = med - ref_df["p25"].to_numpy()
    err_hi = ref_df["p75"].to_numpy()
    ax.barh(y, med, xerr=[err_lo, err_hi], color=BIO_COLOR_CELL, alpha=0.85, capsize=3)
    ax.set_yticks(y)
    ax.set_yticklabels(ref_df["label"])
    ax.set_xlabel("Value (median bar, IQR error bars)")
    ax.set_title("Reference intervals — median and IQR per metric")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    plt.show()
else:
    print("No metrics available for reference intervals.")


## 17) Within- vs Between-Frog Variation & ICC

How much of cell area variation is **between frogs** vs **within frogs**, and how stable are image-level replicates?


In [ ]:
def _icc_oneway(df: pd.DataFrame, group_col: str, value_col: str) -> dict:
    """ICC(1,1) — single measures, groups = frogs."""
    work = df[[group_col, value_col]].dropna()
    work[value_col] = pd.to_numeric(work[value_col], errors="coerce")
    work = work.dropna()
    if work[group_col].nunique() < 2:
        return {"icc": np.nan, "n": len(work), "n_groups": work[group_col].nunique()}

    groups = work.groupby(group_col)[value_col]
    counts = groups.count().to_numpy(dtype=float)
    means = groups.mean()
    grand = work[value_col].mean()
    k = len(counts)
    n = counts.sum()

    ss_between = float(((means - grand) ** 2 * groups.count()).sum())
    ss_within = float(groups.apply(lambda s: ((s - s.mean()) ** 2).sum()).sum())
    df_b, df_w = k - 1, n - k
    ms_b = ss_between / df_b if df_b > 0 else 0.0
    ms_w = ss_within / df_w if df_w > 0 else 0.0
    n_bar = float(counts.mean())
    denom = ms_b + (n_bar - 1) * ms_w
    icc = (ms_b - ms_w) / denom if denom > 0 else np.nan
    var_total = work[value_col].var(ddof=1)
    var_between_frogs = means.var(ddof=1)
    return {
        "icc": float(icc),
        "n": int(n),
        "n_groups": int(k),
        "ms_between": ms_b,
        "ms_within": ms_w,
        "var_total": float(var_total),
        "var_frog_means": float(var_between_frogs),
        "median_within_frog_cv_pct": float(
            groups.apply(lambda s: s.std(ddof=1) / s.mean() * 100 if s.mean() else np.nan).median()
        ),
    }


if "area_um2" in area_df.columns and "frog_id" in area_df.columns:
    icc_stats = _icc_oneway(area_df, "frog_id", "area_um2")
    print("Intraclass correlation (ICC) for cell area_um2 by frog_id:")
    for k, v in icc_stats.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) and np.isfinite(v) else f"  {k}: {v}")

    within_cv = (
        area_df.groupby("frog_id")["area_um2"]
        .apply(lambda s: pd.to_numeric(s, errors="coerce").std(ddof=1) / pd.to_numeric(s, errors="coerce").mean() * 100)
        .dropna()
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), dpi=FIG_DPI)
    axes[0].hist(within_cv, bins=35, color=BIO_COLOR_CELL, alpha=0.85, edgecolor="white")
    axes[0].axvline(within_cv.median(), color="#222", ls="--", label=f"median CV = {within_cv.median():.1f}%")
    axes[0].set_xlabel("Within-frog CV% (area_um2)")
    axes[0].set_ylabel("Number of frogs")
    axes[0].set_title("Cell-to-cell spread within each frog")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.2)

    if "image_path" in area_df.columns:
        img_means = area_df.groupby(["frog_id", "image_path"])["area_um2"].mean().reset_index()
        frogs_multi = img_means.groupby("frog_id").filter(lambda g: len(g) >= 3)["frog_id"].unique()[:12]
        data = [img_means.loc[img_means["frog_id"] == fid, "area_um2"].to_numpy() for fid in frogs_multi]
        axes[1].boxplot(
            data,
            tick_labels=[str(int(f)) for f in frogs_multi],
            vert=True,
        )
        axes[1].set_xlabel("frog_id (≥3 images)")
        axes[1].set_ylabel("Mean cell area per image (µm²)")
        axes[1].set_title("Image-level replicate variation (sample frogs)")
        axes[1].tick_params(axis="x", rotation=45, labelsize=8)
        axes[1].grid(axis="y", alpha=0.2)

    fig.suptitle("Variation structure — within frog vs. between images", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print("area_um2 / frog_id missing; skipping ICC section.")


## 18) Morphology Correlation & PCA

Pairwise correlations among size/shape metrics and a 2D PCA view (top frogs highlighted).


In [ ]:
CORR_COLS = [
    c
    for c in [
        "area_um2",
        "major_axis_um",
        "minor_axis_um",
        "cell_axis_ratio",
        "nucleus_area_um2",
        "nc_ratio",
        "nucleus_axis_ratio",
    ]
    if c in area_df.columns
]

if len(CORR_COLS) >= 2:
    corr_mat = area_df[CORR_COLS].apply(pd.to_numeric, errors="coerce").corr()
    print("Pairwise correlations (good cells):")
    display(corr_mat.round(3))

    fig, ax = plt.subplots(figsize=(7.5, 6), dpi=FIG_DPI)
    im = ax.imshow(corr_mat.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(CORR_COLS)))
    ax.set_yticks(range(len(CORR_COLS)))
    ax.set_xticklabels(CORR_COLS, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(CORR_COLS, fontsize=9)
    for i in range(len(CORR_COLS)):
        for j in range(len(CORR_COLS)):
            ax.text(j, i, f"{corr_mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title("Morphology correlation matrix")
    fig.tight_layout()
    plt.show()

    pca_data = area_df[CORR_COLS].apply(pd.to_numeric, errors="coerce").dropna()
    if len(pca_data) >= 50:
        X = pca_data.to_numpy(dtype=float)
        X = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
        _, svals, vt = np.linalg.svd(X, full_matrices=False)
        scores = X @ vt.T
        var_explained = (svals ** 2) / (svals ** 2).sum() * 100

        fig, ax = plt.subplots(figsize=(7.5, 6), dpi=FIG_DPI)
        frog_ids = area_df.loc[pca_data.index, "frog_id"].to_numpy() if "frog_id" in area_df.columns else None
        if frog_ids is not None:
            top_frogs = pd.Series(frog_ids).value_counts().head(15).index.tolist()
            mask_top = np.isin(frog_ids, top_frogs)
            ax.scatter(scores[~mask_top, 0], scores[~mask_top, 1], s=8, c="#cbd5e1", alpha=0.4, edgecolors="none")
            for fid in top_frogs:
                m = frog_ids == fid
                ax.scatter(scores[m, 0], scores[m, 1], s=14, alpha=0.7, label=f"frog {int(fid)}")
            ax.legend(fontsize=6, loc="best", ncol=2, framealpha=0.9)
        else:
            ax.scatter(scores[:, 0], scores[:, 1], s=10, c=BIO_COLOR_CELL, alpha=0.5)
        ax.set_xlabel(f"PC1 ({var_explained[0]:.1f}% var)")
        ax.set_ylabel(f"PC2 ({var_explained[1]:.1f}% var)")
        ax.set_title("PCA of morphology (top frogs highlighted)")
        ax.grid(alpha=0.2)
        fig.tight_layout()
        plt.show()
        print("PC loadings (rows = variables):")
        loadings = pd.DataFrame(vt[:2].T, index=CORR_COLS, columns=["PC1", "PC2"])
        display(loadings.round(3))
else:
    print("Not enough morphology columns for correlation / PCA.")


## 19) Shape–Size & Nucleus–Cell Shape Coupling

Relationships between cell size, elongation, nucleus shape, and N/C ratio.


In [ ]:
def _eccentricity(major, minor):
    a = pd.to_numeric(major, errors="coerce")
    b = pd.to_numeric(minor, errors="coerce")
    ratio = (b / a).clip(upper=1)
    return np.sqrt(1 - ratio ** 2)


shape_work = area_df.copy()
if {"major_axis_um", "minor_axis_um"}.issubset(shape_work.columns):
    shape_work["cell_eccentricity"] = _eccentricity(
        shape_work["major_axis_um"], shape_work["minor_axis_um"]
    )
if {"nucleus_major_axis_um", "nucleus_minor_axis_um"}.issubset(shape_work.columns):
    shape_work["nucleus_eccentricity"] = _eccentricity(
        shape_work["nucleus_major_axis_um"], shape_work["nucleus_minor_axis_um"]
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=FIG_DPI)
plotted = False

if {"area_um2", "cell_axis_ratio"}.issubset(shape_work.columns):
    x = pd.to_numeric(shape_work["area_um2"], errors="coerce")
    y = pd.to_numeric(shape_work["cell_axis_ratio"], errors="coerce")
    m = x.notna() & y.notna()
    axes[0].scatter(x[m], y[m], s=8, alpha=0.25, c=BIO_COLOR_CELL, edgecolors="none")
    axes[0].set_xlabel("Cell area (µm²)")
    axes[0].set_ylabel("Cell axis ratio (major/minor)")
    axes[0].set_title(f"Size vs. elongation (r = {x[m].corr(y[m]):.3f})")
    axes[0].grid(alpha=0.2)
    plotted = True

if {"cell_axis_ratio", "nucleus_axis_ratio"}.issubset(shape_work.columns):
    x = pd.to_numeric(shape_work["cell_axis_ratio"], errors="coerce")
    y = pd.to_numeric(shape_work["nucleus_axis_ratio"], errors="coerce")
    m = x.notna() & y.notna()
    axes[1].scatter(x[m], y[m], s=8, alpha=0.25, c=BIO_COLOR_NUCLEUS, edgecolors="none")
    axes[1].set_xlabel("Cell axis ratio")
    axes[1].set_ylabel("Nucleus axis ratio")
    axes[1].set_title(f"Cell vs. nucleus shape (r = {x[m].corr(y[m]):.3f})")
    axes[1].grid(alpha=0.2)
    plotted = True

if plotted:
    fig.suptitle("Shape–size coupling in good cells", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
else:
    print("Shape columns missing; skipping shape–size plots.")

if {"area_um2", "nc_ratio"}.issubset(shape_work.columns):
    fig, ax = plt.subplots(figsize=(7, 5), dpi=FIG_DPI)
    x = pd.to_numeric(shape_work["area_um2"], errors="coerce")
    y = pd.to_numeric(shape_work["nc_ratio"], errors="coerce")
    m = x.notna() & y.notna()
    hb = ax.hexbin(x[m], y[m], gridsize=45, cmap="YlGnBu", mincnt=1)
    fig.colorbar(hb, ax=ax, label="count")
    ax.set_xlabel("Cell area (µm²)")
    ax.set_ylabel("N/C ratio")
    ax.set_title(f"N/C ratio vs. cell area (r = {x[m].corr(y[m]):.3f})")
    ax.grid(alpha=0.15)
    fig.tight_layout()
    plt.show()


## 20) Publication-Ready Frog Summary Table


Per-frog table with morphology, classifier yield, and low-`n` flags. Exported to `classify_output/analysis/frog_summary_report.csv` and `frog_summary_report.xlsx`.


In [ ]:
LOW_N_CELLS = 60

if not frog_df.empty and has_col(pred_df, "predicted_verdict"):
    frog_report = frog_df.copy()
    if has_col(pred_df, "frog_id"):
        yld = (
            pred_df.groupby("frog_id", as_index=False)
            .agg(
                n_detected=("mask_index", "count"),
                n_good=("predicted_verdict", lambda s: (s == "good").sum()),
            )
            .assign(good_fraction=lambda d: d["n_good"] / d["n_detected"].clip(lower=1))
        )
        frog_report = frog_report.merge(yld, on="frog_id", how="left")

    if "area_um2_mean" in frog_report.columns and "area_um2_std" in frog_report.columns:
        frog_report["area_um2_cv_pct"] = (
            pd.to_numeric(frog_report["area_um2_std"], errors="coerce")
            / pd.to_numeric(frog_report["area_um2_mean"], errors="coerce")
            * 100
        )

    frog_report["low_n_flag"] = pd.to_numeric(frog_report.get("n_cells", np.nan), errors="coerce") < LOW_N_CELLS

    FROG_REPORT_COLS = [
        "frog_id",
        "n_images",
        "n_cells",
        "n_detected",
        "good_fraction",
        "area_um2_mean",
        "area_um2_std",
        "area_um2_cv_pct",
        "major_axis_um_mean",
        "major_axis_um_std",
        "minor_axis_um_mean",
        "minor_axis_um_std",
        "cell_axis_ratio_mean",
        "cell_axis_ratio_std",
        "nc_ratio_mean",
        "nc_ratio_std",
        "nucleus_area_um2_mean",
        "nucleus_area_um2_std",
        "nucleus_major_axis_um_mean",
        "nucleus_major_axis_um_std",
        "nucleus_minor_axis_um_mean",
        "nucleus_minor_axis_um_std",
        "nucleus_axis_ratio_mean",
        "nucleus_axis_ratio_std",
        "low_n_flag",
    ]
    show_cols = [c for c in FROG_REPORT_COLS if c in frog_report.columns]
    missing = [c for c in FROG_REPORT_COLS if c not in frog_report.columns and c != "area_um2_cv_pct"]
    if missing:
        print("Columns not in frog_df (skipped):", missing)
    frog_report = frog_report[show_cols].sort_values("frog_id")
    print(f"Frog summary table (flag: n_cells < {LOW_N_CELLS}):")
    display(frog_report.head(15))
    display(frog_report.tail(15))

    out_path = ANALYSIS_OUT_DIR / "frog_summary_report.csv"
    frog_report.to_csv(out_path, index=False)
    print("Saved:", out_path.resolve())

    xlsx_path = out_path.with_suffix(".xlsx")
    frog_report.to_excel(xlsx_path, index=False, sheet_name="frog_summary")
    print("Saved:", xlsx_path.resolve())
else:
    print("frog_aggregated_metrics or predictions unavailable; skipping frog summary table.")


## 21) Extended Mixed Models: N/C ~ Cell Size

Tests whether **N/C ratio** decreases with cell area (OLS and mixed-effects with random intercept / slope per frog).


In [ ]:
try:
    import statsmodels.api as sm
    import warnings

    if {"area_um2", "nc_ratio", "frog_id"}.issubset(area_df.columns):
        work = area_df[["area_um2", "nc_ratio", "frog_id"]].dropna()
        work = work[(work["area_um2"] > 0) & (work["nc_ratio"] > 0)].copy()
        work["log_area"] = np.log(work["area_um2"].astype(float))
        work["log_nc"] = np.log(work["nc_ratio"].astype(float))

        ols = sm.OLS.from_formula("log_nc ~ log_area", data=work).fit()
        print("Pooled OLS: log(nc_ratio) ~ log(area_um2)")
        print(ols.summary().tables[1])

        mixed_rows = []
        for label, formula, re_formula in [
            ("random intercept", "log_nc ~ log_area", "1"),
            ("random intercept + slope", "log_nc ~ log_area", "~log_area"),
        ]:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    md = sm.MixedLM.from_formula(
                        formula, data=work, groups="frog_id", re_formula=re_formula
                    )
                    mres = md.fit(method="lbfgs", reml=True, maxiter=200)
                slope = float(mres.fe_params.get("log_area", np.nan))
                ci = mres.conf_int().loc["log_area"] if "log_area" in mres.fe_params.index else [np.nan, np.nan]
                mixed_rows.append(
                    {
                        "model": label,
                        "slope_log_area": slope,
                        "slope_ci_lo": float(ci[0]) if len(ci) else np.nan,
                        "slope_ci_hi": float(ci[1]) if len(ci) else np.nan,
                        "aic": float(mres.aic),
                        "n_cells": len(work),
                        "n_frogs": int(work["frog_id"].nunique()),
                    }
                )
                print(f"\nMixedLM ({label}): slope = {slope:.4f}")
            except Exception as exc:
                print(f"MixedLM ({label}) failed: {exc}")

        if mixed_rows:
            mixed_nc_df = pd.DataFrame(mixed_rows)
            display(mixed_nc_df.round(4))
            mixed_nc_df.to_csv(ANALYSIS_OUT_DIR / "nc_ratio_mixed_models.csv", index=False)

        fig, ax = plt.subplots(figsize=(8, 5.5), dpi=FIG_DPI)
        sample = work.sample(min(8000, len(work)), random_state=RANDOM_SEED)
        ax.scatter(sample["log_area"], sample["log_nc"], s=10, alpha=0.2, c="#cbd5e1", edgecolors="none")
        xx = np.linspace(work["log_area"].min(), work["log_area"].max(), 100)
        ax.plot(xx, ols.params["Intercept"] + ols.params["log_area"] * xx, color="#0f172a", lw=2, label="OLS fit")
        ax.set_xlabel("log(cell area µm²)")
        ax.set_ylabel("log(N/C ratio)")
        ax.set_title("N/C ratio decreases with cell size (log scale)")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        plt.show()

        print(
            "Interpretation: negative slope ⇒ larger cells have proportionally smaller nuclei "
            "(lower N/C ratio)."
        )
    else:
        print("Required columns missing for N/C mixed models.")
except ImportError:
    print("statsmodels not available; skipping section 21.")


## Export figures for LaTeX biology report

Regenerates all figures used by `build_report_latex.py` via the shared `biology_plots` module.
Run from repo root: `python notebooks/build_report_latex.py` for the full PDF pipeline.


In [ ]:
import sys
from pathlib import Path

_nb_dir = Path.cwd()
if (_nb_dir / "biology_plots.py").is_file():
    sys.path.insert(0, str(_nb_dir))
elif (_nb_dir.parent / "notebooks" / "biology_plots.py").is_file():
    sys.path.insert(0, str(_nb_dir.parent / "notebooks"))

from biology_plots import write_all

write_all(area_df=area_df, frog_df=frog_df, pred_df=pred_df)
print("Figures written to notebooks/figures/")
print("Full PDF: python notebooks/build_report_latex.py --no-figures")



## 14) Final Key Findings


In [ ]:
lines = []

n_pred = len(pred_df)
n_area = len(area_df)
lines.append(f"- Loaded **{n_pred}** prediction rows and **{n_area}** filtered-area rows.")

if has_col(pred_df, "predicted_verdict"):
    vc = pred_df["predicted_verdict"].value_counts(dropna=False)
    parts = [f"{k}: {int(v)}" for k, v in vc.items()]
    lines.append("- Verdict distribution: " + ", ".join(parts) + ".")

if has_col(pred_df, "accepted"):
    acc_rate = pd.to_numeric(pred_df["accepted"], errors="coerce").mean()
    if pd.notna(acc_rate):
        lines.append(f"- Accepted-rate estimate: **{acc_rate * 100:.2f}%**.")

if join_keys:
    merged_tmp = pred_df.merge(area_df, on=join_keys, how="left", indicator=True)
    match_rate = (merged_tmp["_merge"] == "both").mean()
    lines.append(f"- Prediction-to-area join match rate (LEFT JOIN on {join_keys}): **{match_rate * 100:.2f}%**.")
else:
    lines.append("- No common join keys were available between predictions and filtered areas.")

if has_col(area_df, "area_px"):
    area_series = pd.to_numeric(area_df["area_px"], errors="coerce")
    med = area_series.median()
    q1 = area_series.quantile(0.25)
    q3 = area_series.quantile(0.75)
    lines.append(f"- `area_px` median: **{med:.2f}** (IQR: {q1:.2f} to {q3:.2f}).")

if "outlier_reports" in globals() and outlier_reports:
    metric_max = max(outlier_reports.items(), key=lambda kv: kv[1]["count"])
    lines.append(
        f"- Highest outlier count observed in **{metric_max[0]}**: {metric_max[1]['count']} rows (IQR method)."
    )

md = "## Key Findings\n" + "\n".join(lines)
display(Markdown(md))
